In [ ]:
from pathlib import Path
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Notebook location:
# BOLT_ISIMIP/src/notebooks
# Move up two levels to BOLT_ISIMIP

BASE = Path.cwd().parents[1]

print("Project Root:")
print(BASE)

In [ ]:
DATA_DIR = (
    BASE
    / "data"
    / "cordex-aus-22"
    / "MPI-M-MPI-ESM-LR_CCLM5-0-15"
)

pr_files = sorted(DATA_DIR.glob("pr*.nc"))

print("Found precipitation files:")

for f in pr_files:
    print(f.name)

INPUT = pr_files[0]

print("\nUsing:")
print(INPUT)

In [ ]:
OUTDIR = BASE / "output"

RX1_DIR = OUTDIR / "rx1day"
RX5_DIR = OUTDIR / "rx5day"
FIG_DIR = OUTDIR / "figures"

RX1_DIR.mkdir(parents=True, exist_ok=True)
RX5_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
ds = xr.open_dataset(INPUT)

ds

In [ ]:
if "pr" in ds.data_vars:
    var = "pr"
else:
    var = list(ds.data_vars)[0]

print("Variable:", var)
print("Units:", ds[var].attrs.get("units"))

-- Compute RX1 Day

In [ ]:
rx1day = ds[var].groupby("time.year").max(dim="time")

rx1day.name = "rx1day"

rx1day.attrs["long_name"] = "Annual Maximum 1-day Precipitation"

rx1day.attrs["units"] = ds[var].attrs.get("units","")

-- Compute RX5Days

In [ ]:
# Rolling 5-day accumulated rainfall
rolling5 = ds[var].rolling(time=5).sum()

# Annual maximum
rx5day = rolling5.groupby("time.year").max(dim="time")

rx5day.name = "rx5day"

rx5day.attrs["long_name"] = "Annual Maximum Consecutive 5-day Precipitation"

rx5day.attrs["units"] = ds[var].attrs.get("units","")

In [ ]:
outfile = RX1_DIR / "RX1day_1979_2100.nc"

rx1day.to_netcdf(outfile)

print(outfile)

In [ ]:
outfile = RX5_DIR / "RX5day_1979_2100.nc"

rx5day.to_netcdf(outfile)

print(outfile)

In [ ]:
year = 1979

plt.figure(figsize=(8,6))

rx1day.sel(year=year).plot(
    cmap="Blues",
    robust=True
)

plt.title(f"RX1day ({year})")

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"RX1day_{year}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
year = 1979

plt.figure(figsize=(8,6))

rx5day.sel(year=year).plot(
    cmap="Blues",
    robust=True
)

plt.title(f"RX5day ({year})")

plt.tight_layout()

plt.savefig(
    FIG_DIR / f"RX5day_{year}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

-- Time seried Plots

In [ ]:
rx1_mean = rx1day.mean(dim=["lat","lon"])

plt.figure(figsize=(12,5))

rx1_mean.plot(marker="o")

plt.grid(True)

plt.title("Annual Mean RX1day")

plt.xlabel("Year")

plt.ylabel(rx1day.attrs["units"])

plt.show()